# 第 6 关 · 验尸官

长训练，然后把结果拆开看。恢复率是一个数，它掩盖了很多东西。

**BOSS：恢复率 ≥ 第 5 关，且完成点特征消融**

---

**这一关有 6 个部分：**

0. 准备 + 长训练
1. 第一刀：茎区 vs 环区
2. 第二刀：哪种碱基学不会
3. 第三刀：温度
4. 消融：点特征到底有多重要
5. BOSS 与自测

## 部分 0 · 准备 + 长训练

150 轮，大约半小时。跑着的时候往下读，想清楚今天要拆的三刀。
已经有权重就直接读盘，不重训。

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / "minif").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print("仓库根：", ROOT)

from minif.plotting import setup
plt = setup()

In [ ]:
import os
import numpy as np
import torch

from minif import data, engine, features as F, pairing, paths, scoreboard
from minif.model import MiniMPNN
from minif.plotting import tidy, seq_cmap, PALETTE

MOL = "rna"
tr = data.load(MOL, "train"); va = data.load(MOL, "val")
data.describe(tr, "训练集："); data.describe(va, "验证集：")

mk = lambda **kw: MiniMPNN(d_node=F.node_dim(MOL), d_edge=F.edge_dim(MOL),
                           n_letters=F.n_letters(MOL), d_model=128, d_hidden=128,
                           n_enc=3, n_dec=3, **kw)
CKPT = paths.run_file("level6.pt")
print("权重落点", CKPT, "  已存在" if os.path.exists(CKPT) else "  需要训练")

In [ ]:
if not os.path.exists(CKPT):
    engine.fit(mk(), tr, va, MOL, epochs=150, lr=1e-3, noise=0.02,
               autoregressive=True, log_every=10, save=CKPT, level=6, eval_pairs=True)

model = mk(); model.load_state_dict(torch.load(CKPT)); model.eval()
if not scoreboard.read()["levels"].get("6", {}).get("recovery"):
    _, acc = engine.run_epoch(model, va, None, MOL, autoregressive=True)
    pv, _ = engine.eval_pair_validity(model, va, MOL)
    scoreboard.record(6, recovery=acc, pair_validity=pv)
print("就绪")

## 部分 1 · 第一刀：茎区 vs 环区

茎区恢复率应该明显高于环区。为什么？**配对把可选项压到了几种组合，
环区几乎什么都能放。**

这是结构生物学结论，不是机器学习结论——面试时特别好用。

In [ ]:
hit_s = n_s = hit_l = n_l = 0
conf = np.zeros((4, 4), dtype=int)
with torch.no_grad():
    for r in va:
        V, E, idx, seq = engine.prep(r, MOL)
        pred = model(V, E, idx, seq, engine.random_order(len(seq))).argmax(-1)
        hit = (pred == seq).numpy(); paired = r["partner"] >= 0
        hit_s += hit[paired].sum(); n_s += paired.sum()
        hit_l += hit[~paired].sum(); n_l += (~paired).sum()
        for t, p in zip(seq.tolist(), pred.tolist()):
            conf[t, p] += 1

rs, rl = hit_s/max(n_s, 1), hit_l/max(n_l, 1)
print("茎区（配对）  恢复 %.1f%%   n=%d" % (100*rs, n_s))
print("环区（不配对）恢复 %.1f%%   n=%d" % (100*rl, n_l))

In [ ]:
fig, ax = plt.subplots(figsize=(5.2, 4))
vals = [100*rs, 100*rl]
ax.bar(["茎区\n(配对)", "环区\n(不配对)"], vals, color=PALETTE[:2], width=.55)
for x, v in enumerate(vals):
    ax.annotate("%.1f%%" % v, (x, v), xytext=(0, 4), textcoords="offset points",
                ha="center", fontsize=14)
ax.set_ylabel("恢复率 %"); ax.set_title("配对把可选项压窄了")
ax.set_ylim(0, max(vals)*1.25); tidy(ax); plt.show()

## 部分 2 · 第二刀：哪种碱基学不会

混淆矩阵。行是真实碱基，列是模型猜的。对角线是猜对。

In [ ]:
A = F.alphabet(MOL)
norm = conf / conf.sum(1, keepdims=True).clip(1)
fig, ax = plt.subplots(figsize=(5.4, 4.8))
im = ax.imshow(norm, cmap=seq_cmap(), vmin=0, vmax=norm.max())
ax.set_xticks(range(4), list(A)); ax.set_yticks(range(4), list(A))
ax.set_xlabel("模型猜的"); ax.set_ylabel("真实")
ax.set_title("混淆矩阵（每行归一）")
for i in range(4):
    for j in range(4):
        ax.text(j, i, "%.0f%%" % (100*norm[i, j]), ha="center", va="center",
                fontsize=13, color="white" if norm[i, j] > norm.max()*.55 else "#0b0b0b")
ax.set_xticks(np.arange(-.5, 4, 1), minor=True)
ax.set_yticks(np.arange(-.5, 4, 1), minor=True)
ax.grid(which="minor", color="#fcfcfb", lw=2); ax.tick_params(which="minor", length=0)
plt.show()
for i in range(4):
    w = conf[i].copy(); w[i] = -1
    print("  %s  n=%5d  恢复 %4.1f%%   最常被错认成 %s"
          % (A[i], conf[i].sum(), 100*conf[i, i]/max(conf[i].sum(), 1), A[int(w.argmax())]))

## 部分 3 · 第三刀：温度

温度低 = 每次都挑最可能的那个，像天然但几条设计几乎一样。
温度高 = 更多样，但配对更容易出错。

**设计要的不是最像天然，而是在能折回目标骨架的前提下尽量多样。**

In [ ]:
rows = []
for T in (0.1, 0.3, 0.5, 1.0):
    sims, divs, pvs = [], [], []
    for r in va[:6]:
        outs = [engine.design(model, r, MOL, temperature=T) for _ in range(3)]
        sims += [sum(a == b for a, b in zip(r["seq"], o))/len(o) for o in outs]
        divs += [1 - sum(a == b for a, b in zip(outs[x], outs[y]))/len(outs[0])
                 for x in range(3) for y in range(x+1, 3)]
        if (r["partner"] >= 0).any():
            pvs += [pairing.pair_validity(o, r["partner"])[0] for o in outs]
    rows.append((T, np.mean(sims), np.mean(divs), np.nanmean(pvs) if pvs else np.nan))
    print("T=%.1f  与天然相同 %.1f%%   互相不同 %.1f%%   配对有效 %.1f%%"
          % (T, 100*rows[-1][1], 100*rows[-1][2], 100*rows[-1][3]))

In [ ]:
T, sim, div, pv = map(np.array, zip(*rows))
fig, ax = plt.subplots(figsize=(7.4, 4.4))
for y, lab, c in [(sim, "与天然相同", PALETTE[0]), (div, "彼此差异", PALETTE[1]),
                  (pv, "配对有效", PALETTE[2])]:
    ax.plot(T, 100*y, marker="o", color=c, label=lab)
    ax.annotate(lab, (T[-1], 100*y[-1]), xytext=(7, 0), textcoords="offset points",
                va="center", fontsize=13, color=c)
ax.set_xlabel("采样温度"); ax.set_ylabel("%")
ax.set_title("温度：像天然、多样性、配对正确，三者的权衡")
ax.set_xlim(0.05, 1.28); tidy(ax); plt.show()

## 部分 4 · 消融：点特征到底有多重要

把那 9 维点特征全部清零重训。**ProteinMPNN 原文的点特征就是全零的**，
信息全放在边上。掉几个点？

In [ ]:
ab = engine.fit(mk(use_node_feats=False), tr, va, MOL, epochs=60, lr=1e-3, noise=0.02,
                autoregressive=True, log_every=20,
                save=paths.run_file("level6_ablation.pt"), eval_pairs=False)
scoreboard.record(6, ablation_recovery=ab)
full = scoreboard.read()["levels"]["6"]["recovery"]
print("\n完整 %.1f%%  ->  点特征清零 %.1f%%   掉了 %.1f 个百分点"
      % (100*full, 100*ab, 100*(full-ab)))
print("掉得少，说明几何信息本来就几乎全在边上。")

## 部分 5 · BOSS 与自测

In [ ]:
!cd .. && python game.py check 6

## 自测三问

1. 恢复率高是不是就说明设计成功了？用今天的温度表给答案补一句**实测数据**。
2. 自洽性验证是什么？为什么它才是真正的判据？
3. 点特征清零掉几个点？这说明几何信息主要在哪？

写进 `notes/level6.md`。